# 01_validate_raw

This notebook reads raw Olist data from ADLS, validates all required datasets, creates a validation report, and stops execution if any CRITICAL issue is found.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

In [0]:
storage_account_name = "shopscope2026"
container_name = "raw"

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"

dataset_paths = {
    "customers": f"{base_path}/customers/olist_customers_dataset.csv",
    "geolocation": f"{base_path}/geolocation/olist_geolocation_dataset.csv",
    "order_items": f"{base_path}/order_items/olist_order_items_dataset.csv",
    "orders": f"{base_path}/orders/olist_orders_dataset.csv",
    "payments": f"{base_path}/payments/olist_order_payments_dataset.csv",
    "products": f"{base_path}/products/olist_products_dataset.csv",
    "reviews": f"{base_path}/reviews/olist_order_reviews_dataset.csv",
    "sellers": f"{base_path}/sellers/olist_sellers_dataset.csv",
    "category_names": f"{base_path}/category_names/product_category_name_translation.csv"
}

validation_output_path = "abfss://processed@shopscope2026.dfs.core.windows.net/validation_report"

In [0]:
# Pull credentials from Databricks secret scope instead of hardcoding them
client_id = dbutils.secrets.get(scope="shopscope-kv", key="client-id")
tenant_id = dbutils.secrets.get(scope="shopscope-kv", key="tenant-id")
client_secret = dbutils.secrets.get(scope="shopscope-kv", key="client-secret")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("ADLS Authentication Configured")

ADLS Authentication Configured


In [0]:
display(dbutils.fs.ls(base_path))

path,name,size,modificationTime
abfss://raw@shopscope2026.dfs.core.windows.net/category_names/,category_names/,0,1781957230000
abfss://raw@shopscope2026.dfs.core.windows.net/customers/,customers/,0,1781957186000
abfss://raw@shopscope2026.dfs.core.windows.net/geolocation/,geolocation/,0,1781957907000
abfss://raw@shopscope2026.dfs.core.windows.net/order_items/,order_items/,0,1781957175000
abfss://raw@shopscope2026.dfs.core.windows.net/orders/,orders/,0,1781957148000
abfss://raw@shopscope2026.dfs.core.windows.net/payments/,payments/,0,1781957206000
abfss://raw@shopscope2026.dfs.core.windows.net/products/,products/,0,1781957196000
abfss://raw@shopscope2026.dfs.core.windows.net/reviews/,reviews/,0,1781957845000
abfss://raw@shopscope2026.dfs.core.windows.net/sellers/,sellers/,0,1781957943000


In [0]:
datasets = {
    name: spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    for name, path in dataset_paths.items()
}

for name, df in datasets.items():
    print(name, df.count(), len(df.columns))

customers 99441 5
geolocation 1000163 5
order_items 112650 7
orders 99441 8
payments 103886 5
products 32951 9
reviews 104162 7
sellers 3095 4
category_names 71 2


In [0]:
REQUIRED_COLUMNS = {
    "customers": ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"],
    "geolocation": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"],
    "orders": ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
    "payments": ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"],
    "products": ["product_id", "product_category_name"],
    "reviews": ["review_id", "order_id", "review_score"],
    "sellers": ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"],
    "category_names": ["product_category_name", "product_category_name_english"]
}

NULL_THRESHOLDS = {
    "customers": 0.10,
    "geolocation": 0.10,
    "order_items": 0.10,
    "orders": 0.10,
    "payments": 0.10,
    "products": 0.20,
    "reviews": 0.20,
    "sellers": 0.10,
    "category_names": 0.10
}

VALID_ORDER_STATUSES = [
    "created", "approved", "invoiced", "processing",
    "shipped", "delivered", "unavailable", "canceled"
]

MIN_PRICE = 0
MAX_PRICE = 100000

In [0]:
OPTIONAL_COLUMNS = {
    "reviews": {"review_comment_title", "review_comment_message"}
}

def issue(dataset, rule, column, severity, detail):
    return {
        "dataset": dataset,
        "rule": rule,
        "column": column,
        "severity": severity,
        "detail": detail
    }

def check_required_columns(name, df):
    issues = []
    for col in REQUIRED_COLUMNS.get(name, []):
        if col not in df.columns:
            issues.append(issue(name, "required_column_missing", col, "CRITICAL", f"Column '{col}' missing"))
    return issues

def check_null_percentages(name, df):
    issues = []
    total_rows = df.count()
    threshold = NULL_THRESHOLDS.get(name, 0.10)
    optional_cols = OPTIONAL_COLUMNS.get(name, set())

    if total_rows == 0:
        issues.append(issue(name, "empty_dataset", "all", "CRITICAL", "Dataset has 0 rows"))
        return issues

    for col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        pct = null_count / total_rows

        if pct > threshold:
            if col in optional_cols:
                severity = "WARNING"
            else:
                severity = "CRITICAL" if pct > 0.5 else "WARNING"

            issues.append(
                issue(
                    name,
                    "high_null_percentage",
                    col,
                    severity,
                    f"{pct:.1%} null (threshold {threshold:.0%})"
                )
            )
    return issues

def check_duplicates(name, df):
    issues = []
    total_rows = df.count()
    dupes = total_rows - df.distinct().count()

    if dupes > 0:
        pct = dupes / total_rows
        issues.append(
            issue(
                name,
                "duplicate_rows",
                "all",
                "WARNING",
                f"{dupes} duplicate rows ({pct:.1%} of dataset)"
            )
        )
    return issues

def check_price_ranges(name, df):
    issues = []
    for col in [c for c in ["price", "freight_value", "payment_value"] if c in df.columns]:
        num_col = F.expr(f"try_cast({col} as double)")

        malformed = df.filter(F.col(col).isNotNull() & num_col.isNull()).count()
        below = df.filter(num_col.isNotNull() & (num_col < MIN_PRICE)).count()
        above = df.filter(num_col.isNotNull() & (num_col > MAX_PRICE)).count()

        if malformed > 0:
            issues.append(issue(name, "invalid_numeric_value", col, "WARNING", f"{malformed} malformed numeric values"))
        if below > 0:
            issues.append(issue(name, "price_below_minimum", col, "WARNING", f"{below} rows with {col} < {MIN_PRICE}"))
        if above > 0:
            issues.append(issue(name, "price_above_maximum", col, "WARNING", f"{above} rows with {col} > {MAX_PRICE}"))
    return issues

def check_review_scores(name, df):
    issues = []
    if "review_score" not in df.columns:
        return issues

    score_col = F.expr("try_cast(review_score as int)")

    malformed = df.filter(F.col("review_score").isNotNull() & score_col.isNull()).count()
    out_of_range = df.filter(score_col.isNotNull() & (~score_col.isin([1, 2, 3, 4, 5]))).count()

    if malformed > 0:
        issues.append(issue(name, "invalid_review_score", "review_score", "WARNING", f"{malformed} malformed review_score values"))
    if out_of_range > 0:
        issues.append(issue(name, "invalid_review_score", "review_score", "WARNING", f"{out_of_range} scores not in [1,2,3,4,5]"))
    return issues

def check_order_status(name, df):
    issues = []
    if "order_status" not in df.columns:
        return issues

    bad_status = (
        df.filter(
            F.col("order_status").isNotNull() &
            (~F.col("order_status").isin(VALID_ORDER_STATUSES))
        )
        .select("order_status")
        .distinct()
        .collect()
    )

    bad_status = [r["order_status"] for r in bad_status]

    if bad_status:
        issues.append(issue(name, "invalid_order_status", "order_status", "WARNING", f"Unexpected values: {bad_status}"))
    return issues

def validate_dataset(name, df):
    issues = []
    issues.extend(check_required_columns(name, df))
    issues.extend(check_null_percentages(name, df))
    issues.extend(check_duplicates(name, df))
    issues.extend(check_price_ranges(name, df))
    issues.extend(check_review_scores(name, df))
    issues.extend(check_order_status(name, df))
    return issues

In [0]:
all_issues = []

for name, df in datasets.items():
    print(f"Validating {name}...")
    dataset_issues = validate_dataset(name, df)
    all_issues.extend(dataset_issues)
    print(f"{name}: {len(dataset_issues)} issues found")

report_pd = pd.DataFrame(all_issues) if all_issues else pd.DataFrame(
    columns=["dataset", "rule", "column", "severity", "detail"]
)

display(report_pd)

Validating customers...
customers: 0 issues found
Validating geolocation...
geolocation: 1 issues found
Validating order_items...
order_items: 0 issues found
Validating orders...
orders: 0 issues found
Validating payments...
payments: 0 issues found
Validating products...
products: 0 issues found
Validating reviews...
reviews: 5 issues found
Validating sellers...
sellers: 0 issues found
Validating category_names...
category_names: 0 issues found


dataset,rule,column,severity,detail
geolocation,duplicate_rows,all,WARNING,261831 duplicate rows (26.2% of dataset)
reviews,high_null_percentage,review_comment_title,WARNING,88.5% null (threshold 20%)
reviews,high_null_percentage,review_comment_message,WARNING,60.6% null (threshold 20%)
reviews,duplicate_rows,all,WARNING,85 duplicate rows (0.1% of dataset)
reviews,invalid_review_score,review_score,WARNING,2557 malformed review_score values
reviews,invalid_review_score,review_score,WARNING,"1 scores not in [1,2,3,4,5]"


In [0]:
report_spark = spark.createDataFrame(report_pd)

(
    report_spark
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(validation_output_path)
)

print(f"Validation report saved to: {validation_output_path}")

Validation report saved to: abfss://processed@shopscope2026.dfs.core.windows.net/validation_report


In [0]:
critical_count = 0 if report_pd.empty else (report_pd["severity"] == "CRITICAL").sum()
warning_count = 0 if report_pd.empty else (report_pd["severity"] == "WARNING").sum()

print(f"CRITICAL issues: {critical_count}")
print(f"WARNING issues: {warning_count}")

if critical_count > 0:
    raise Exception(f"Validation failed with {critical_count} CRITICAL issues. Review validation_report before proceeding.")
else:
    print("Validation passed. Safe to proceed to 02_build_master_transactions.")

CRITICAL issues: 0
WARNING issues: 6
Validation passed. Safe to proceed to 02_build_master_transactions.


In [0]:
display(report_pd.sort_values(["severity", "dataset", "rule"]))

dataset,rule,column,severity,detail
geolocation,duplicate_rows,all,WARNING,261831 duplicate rows (26.2% of dataset)
reviews,duplicate_rows,all,WARNING,85 duplicate rows (0.1% of dataset)
reviews,high_null_percentage,review_comment_title,WARNING,88.5% null (threshold 20%)
reviews,high_null_percentage,review_comment_message,WARNING,60.6% null (threshold 20%)
reviews,invalid_review_score,review_score,WARNING,2557 malformed review_score values
reviews,invalid_review_score,review_score,WARNING,"1 scores not in [1,2,3,4,5]"


In [0]:
critical_df = report_pd[report_pd["severity"] == "CRITICAL"]

if len(critical_df) > 0:
    display(critical_df)
else:
    print("No CRITICAL issues found.")

No CRITICAL issues found.
